In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install openai

In [ ]:
# libraries
import requests
import os
import ast
import pickle
import pandas as pd
from tqdm.notebook import tqdm
from openai import OpenAI

In [ ]:
# load dataset
review = pd.read_csv("/your_path/sample_review.csv")

# 왜 때문인지 문자열로 저장된 컬럼 리스트로 변환..
review['keywords_bg'] = review['keywords_bg'].apply(lambda x : ast.literal_eval(x))
result = []

In [ ]:
file_path = '/your_path_to_save'
api_key = 'your_key'

# 중간에 끊기면 pickle 불러와서 다시 시작
# with open(file_path+'/topic.pkl', 'rb') as file :
#   result = pickle.load(file)

existing_ids = set(list(res.keys())[0] for res in result)

with tqdm(total=len(review)) as pbar :
  for idx, row in review.iterrows() :
    if idx < len(result) :
      pbar.update(1)
      continue
    else:
      prompt = f"""Keyword List: {set(row['keywords_bg'])}
          These keywords are related to the review of restaurants or food category.
          You have to work on the following requirements:
          1. Group keywords in Keyword List based on similarity. *Every keyword must belong to a topic*.
          2. Name each group and make it the upper topic.
          3. Consider fixed topics for suggested categories, like 'food', 'service', 'atmosphere', 'facility', 'price', and 'others'.
            - food: Keywords related to menu items to eat, taste, beverages, and nutritional values.
            - service: Keywords related to customer comport and assistance, delivery(i.e. 'deli', 'uber'), utilities, appropriateness, and courtesy in operation.
            - atmosphere: Keywords related to the overall dining environment and mood, including interior design, furniture, lighting, and outdoor spaces.
            - facility: Keywords related to the physical amenities of the restaurant, such as cleanliness, parking, restrooms and equipment.
            - price: Keywords related to the cost, pricing, value for money, affordability, and overall expense of the dining experience.
            - others: Keywords that don't fit into the above categories, especially those related to store names, reviews, or star ratings.
          4. After naming, return the text in Python dictionary format: keys are topics, values are keywords. *Set values in list type*
          5. If some topic hasn't any keyword, *Don't include that topic key in dictionary*.
          """

      client = OpenAI(api_key=api_key)

      response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        response_format={ "type": "text" },
        messages=[
            {"role": "user", "content": prompt},])

      temp = response.choices[0].message.content.strip()
      result.append({row['review_id']: temp})
      # print(temp)  # 잘 뽑히는지 확인

      # 20개씩 저장
      if len(result)%20 == 0 :
        with open(file_path + '/file_anme.pkl', 'wb') as file :
          pickle.dump( result, file)
      pbar.update(1)
  # 끝나고 저장
  with open(file_path + '/file_name.pkl', 'wb') as file :
    pickle.dump( result, file)